In [1]:
import matplotlib
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt
import os
import numpy as np
from fairchem.core.datasets import AseDBDataset
from ase.calculators.emt import EMT 
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from fairchem.core import pretrained_mlip, FAIRChemCalculator
from ase.io.trajectory import Trajectory
from ase import units
from ase.io import read
from ase.db import connect
from ase.calculators.orca import ORCA, OrcaProfile

/opt/homebrew/Caskroom/miniconda/base/envs/SLAC/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-14 09:52:28,713	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
W0414 09:52:29.294000 8297 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
dataset_path = "/Users/joaopanicali/SLAC_v2/SLAC-atoms/train_4M" # using 4m dataset for exploration. Much smaller than the whole dataset
dataset = AseDBDataset({"src": dataset_path})
print(f"Total molecules in this split: {len(dataset)}")

Total molecules in this split: 3986754


In [3]:
def parse_orca_hessian(filepath: str) -> np.ndarray:
    """Parses an ORCA .hess file and extracts the symmetric Hessian matrix."""
    with open(filepath, 'r') as f:
        lines = f.readlines()

    start_line = next((i + 2 for i, line in enumerate(lines) if "$hessian" in line), None)
    if start_line is None:
        raise ValueError("Hessian section not found in the .hess file.")

    dim = int(lines[start_line - 1].strip())
    hessian = np.zeros((dim, dim))
    i = start_line
    
    while i < len(lines):
        parts = lines[i].split()
        if not parts or "$" in lines[i]: 
            break
        if not any("." in p for p in parts):
            col_start = int(parts[0])
        else:
            row_idx = int(parts[0])
            for j, val in enumerate(map(float, parts[1:])):
                hessian[row_idx, col_start + j] = val
        i += 1

    return (hessian + hessian.T) / 2

In [57]:
def compute_and_save_modes(atoms, orca_path: str, output_npz_path: str, scratch_dir: str = "./orca_scratch"):
    """
    Runs ORCA, extracts the Hessian, calculates normal modes using ASE Standard Units, 
    and saves the physics data to disk for rapid trajectory generation.
    """
    os.makedirs(scratch_dir, exist_ok=True)
    base_name = os.path.join(scratch_dir, "mol_opt_freq")
    
    # 1. Run Quantum Chemistry (Optimization + Frequencies)
    atoms.calc = ORCA(
        profile=OrcaProfile(command=orca_path),
        directory="./orca_scratch",  
        label="mol_opt_freq",        
        orcasimpleinput='wB97M-D3BJ def2-TZVPD Opt Freq', 
        
        # Notice: %pal nprocs 8 end is GONE.
        # Maxcore is now back to default or a safe 4000 since it has the whole machine to itself
        orcablocks='%maxcore 4000\n%scf MaxIter 300 end',
        
        charge=atoms.info["charge"],
        mult=atoms.info["spin"], 
    )
    
    atoms.get_potential_energy() 
    
    # 2. Extract Geometry (Already in Angstroms) and Masses (Already in amu)
    R_0 = atoms.get_positions().flatten() 
    masses_amu = np.repeat(atoms.get_masses(), 3) 
    
    # 3. Load Hessian and Convert to ASE Units (eV / Angstrom^2)
    raw_hessian = parse_orca_hessian(f"/Users/joaopanicali/SLAC_v2/SLAC-atoms/orca_scratch/orca.hess")
    
    # ORCA outputs Hartree/Bohr^2. ASE converts this to eV/Angstrom^2
    conversion_factor = units.Hartree / (units.Bohr ** 2)
    hessian_ASE = raw_hessian * conversion_factor
    
    # 4. Mass-Weighting and Eigendecomposition
    M_inv_sqrt = np.diag(1.0 / np.sqrt(masses_amu))
    tilde_H_ASE = M_inv_sqrt @ hessian_ASE @ M_inv_sqrt
    
    eigenvalues, V = np.linalg.eigh(tilde_H_ASE)
    
    # 5. Isolate Vibrations 
    # We sort by absolute value to cleanly drop the 6 rigid body modes (~0.0), 
    # ensuring we don't accidentally drop imaginary frequencies if Opt failed.
    n_atoms = len(atoms)
    if n_atoms == 1:
        n_rigid = 3
    elif n_atoms == 2:
        n_rigid = 5
    else:
        n_rigid = 6
    vib_indices = np.argsort(np.abs(eigenvalues))[n_rigid:]
    
    # Take absolute value before sqrt to prevent NaN crashes from numerical noise (-1e-8)
    omegas_ASE = np.sqrt(np.abs(eigenvalues[vib_indices])) 
    V_vib = V[:, vib_indices] 
    
    # 6. Save to compressed disk format
    np.savez_compressed(
        output_npz_path,
        R_0=R_0,
        masses_amu=masses_amu,
        omegas_ASE=omegas_ASE,
        V_vib=V_vib
    )
    print(f"Physical mode data successfully saved to {output_npz_path}")

In [42]:
shard_path = "/Users/joaopanicali/SLAC_v2/SLAC-atoms/train_4M/data0000.aselmdb"

print("Connecting to LMDB Shard 0...")
db = connect(shard_path)

print(f"Total molecules in this single shard: {len(db)}")
print("Hunting for the smallest molecule...\n")

# Query just this shard, sort by atom count, and grab the smallest one
for row in db.select(limit=1, sort='natoms'):
    print(f"--- SUCCESS: TINY MOLECULE FOUND ---")
    print(f"Formula: {row.formula}")
    print(f"Atom Count: {row.natoms}")
    print(f"Charge: {row.data.get('charge', 0)}")
    print(f"Spin/Mult: {row.data.get('spin', 1)}")
    
    # We bypass the PyTorch dataset index and pull the 3D atoms object straight out!
    atoms = row.toatoms()
    
    # Manually attach the info dictionary so our ORCA pipeline can read it
    atoms.info["charge"] = row.data.get("charge", 0)
    atoms.info["spin"] = row.data.get("spin", 1)
    
    break

Connecting to LMDB Shard 0...
Total molecules in this single shard: 49835
Hunting for the smallest molecule...

--- SUCCESS: TINY MOLECULE FOUND ---
Formula: TiCl
Atom Count: 2
Charge: 0
Spin/Mult: 4


In [64]:
orca_command_path = "/Users/joaopanicali/software/orca_6/orca"
output_physics_file = "/Users/joaopanicali/SLAC_v2/SLAC-atoms/mol_001_physics.npz"
compute_and_save_modes(
    atoms=atoms, 
    orca_path=orca_command_path, 
    output_npz_path=output_physics_file,
    scratch_dir="./orca_scratch"
)

              geometry optimization, check very carefully.
              ORCA does not by default supply the forces for the
              converged geometry!
Physical mode data successfully saved to /Users/joaopanicali/SLAC_v2/SLAC-atoms/mol_001_physics.npz


In [65]:
def generate_trajectory(npz_path: str, num_frames: int = 1000, T_kelvin: float = 300.0, duration_ps: float = 1.0) -> np.ndarray:
    """
    Loads saved physical mode data and generates a synthetic 3D trajectory tensor
    using pure ASE Standard Units for flawless numerical stability.
    """
    # 1. Load Data
    data = np.load(npz_path)
    R_0 = data['R_0']
    masses_amu = data['masses_amu']
    omegas_ASE = data['omegas_ASE']
    V_vib = data['V_vib']
    
    num_modes = len(omegas_ASE)
    N_atoms = len(R_0) // 3
    
    # 2. Statistical Mechanics (Equipartition Theorem)
    # Total vibrational energy is k_B * T (1/2 potential, 1/2 kinetic).
    # E_total = 1/2 * omega^2 * A^2 = k_B * T --> A = sqrt(2 * k_B * T) / omega
    # Note: units.kB is exactly the Boltzmann constant in eV/K.
    A_q = np.sqrt(2 * units.kB * T_kelvin) / omegas_ASE 
    
    # 3. Vectorized Simulation 
    # Convert input duration (picoseconds) into femtoseconds, then into ASE time units.
    # units.fs represents the value of 1 femtosecond in ASE time.
    duration_fs = duration_ps * 1000
    time_steps_ASE = np.linspace(0, duration_fs * units.fs, num_frames).reshape(-1, 1) 
    
    # Random initial phases
    phi = np.random.uniform(0, 2 * np.pi, num_modes) 
    
    # Calculate modes over time. 
    # c_t shape: (num_frames, 3N-6)
    c_t = A_q * np.cos(omegas_ASE * time_steps_ASE + phi) 
    
    # Superposition in mass-weighted space. 
    # q_t shape: (3N, num_frames)
    q_t = V_vib @ c_t.T 
    
    # 4. Un-weight Masses 
    # Because q_t is in sqrt(amu * Angstrom^2), multiplying by 1/sqrt(amu) 
    # perfectly leaves delta_R naturally formatted in Angstroms!
    M_inv_sqrt = np.diag(1.0 / np.sqrt(masses_amu))
    delta_R_Angstroms = M_inv_sqrt @ q_t 
    
    # 5. Add to equilibrium geometry and format
    R_t = R_0[:, None] + delta_R_Angstroms 
    trajectory_tensor = R_t.T.reshape(num_frames, N_atoms, 3)
    
    return trajectory_tensor

In [66]:
generate_trajectory(output_physics_file)

/var/folders/nn/18cpry7128317g4gmfl08t8r0000gn/T/ipykernel_8297/1048074374.py:43: RuntimeWarning: divide by zero encountered in matmul
  delta_R_Angstroms = M_inv_sqrt @ q_t
/var/folders/nn/18cpry7128317g4gmfl08t8r0000gn/T/ipykernel_8297/1048074374.py:43: RuntimeWarning: overflow encountered in matmul
  delta_R_Angstroms = M_inv_sqrt @ q_t
/var/folders/nn/18cpry7128317g4gmfl08t8r0000gn/T/ipykernel_8297/1048074374.py:43: RuntimeWarning: invalid value encountered in matmul
  delta_R_Angstroms = M_inv_sqrt @ q_t


array([[[-19.59382,  15.88798,  14.14857],
        [-17.79214,  17.95411,  13.02849]],

       [[-19.59382,  15.88798,  14.14857],
        [-17.79214,  17.95411,  13.02849]],

       [[-19.59382,  15.88798,  14.14857],
        [-17.79214,  17.95411,  13.02849]],

       ...,

       [[-19.59382,  15.88798,  14.14857],
        [-17.79214,  17.95411,  13.02849]],

       [[-19.59382,  15.88798,  14.14857],
        [-17.79214,  17.95411,  13.02849]],

       [[-19.59382,  15.88798,  14.14857],
        [-17.79214,  17.95411,  13.02849]]], shape=(1000, 2, 3))